# CS04-A — Exploratory feature-combination matrix with return magnitude

**Historical discovery only, not independent confirmation or AM-T-HAR-Blend integration.** Ten groups constructed from previously downloaded Binance spot 5m and USD-M perpetual 1m data. All 175 feature-family subsets of size one to three are added to a shared sign/streak/realized-variance baseline. A three-fold expanding walk-forward scan fits multinomial DOWN/ZERO/UP probabilities and separate mean-absolute-net-return forecasts at 5, 15 and 30 minutes.

**Signed future net log returns and their absolute, squared, upside and downside components are retained** in the local-only Parquet feature/label store for future AM-T-HAR-Blend assessment. The script NEVER places future magnitude in predictors. No pair/triple nonlinear interaction terms in CS04-A; this is an additive feature-subset sweep.

See [the frozen CS04-A protocol](../../discussions/Candle_Streaks/004a_chatgpt_feature_matrix_preregistration.md). Previous CS03/CS03b TEST dates are **not** used for ranking.

## 0 — Environment and synthetic unit tests

In [1]:
import os,sys,json,subprocess,time,gc,tempfile
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Execute inside a project checkout'
load_dotenv(ROOT/'.env')
sys.path.insert(0,str(ROOT/'src'))
from data_engineering.modeling import candle_streak_feature_matrix as fm
from data_engineering.modeling import markov_lift_distributions as vd14
cfg=fm.Specification()
SYNTHETIC=os.getenv('CS04A_SYNTHETIC','0')=='1'
QUICK=os.getenv('CS04A_QUICK','0')=='1'
if SYNTHETIC:
    tmp=tempfile.TemporaryDirectory(prefix='cs04a-')
    OUT=Path(tmp.name)/'outputs'; LOCAL=Path(tmp.name)/'features'
else:
    rootval=os.getenv('DERIBIT_DATA_ROOT')
    if not rootval:raise RuntimeError('Set DERIBIT_DATA_ROOT in .env')
    DATA_ROOT=Path(rootval)
    BINANCE=Path(os.environ.get('BINANCE_UM_DIR') or DATA_ROOT.parent/'Binance')
    SPOT=DATA_ROOT/'history'/'price_5m'
    OUT=ROOT/'notebooks'/'Candle_Streaks'/'04a'
    LOCAL=DATA_ROOT/'research'/'cs04a_feature_store'
OUT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)
print(fm.VERSION,'synthetic=',SYNTHETIC,'quick=',QUICK,'combinations=',len(fm.combinations_frozen()))
print('Outputs:',OUT,'local-only labels:',LOCAL)
test_env=os.environ.copy();test_env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+test_env.get('PYTHONPATH','')
test=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_candle_streak_feature_matrix.py')],env=test_env,cwd=ROOT,capture_output=True,text=True)
print(test.stdout[-1800:]);print(test.stderr[-500:])
assert test.returncode==0,'CS04-A regression tests failed: stop before historical scan' 

CS04A-0.1.0 synthetic= False quick= False combinations= 175
Outputs: D:\Projects\deribit_market_data\notebooks\Candle_Streaks\04a local-only labels: D:\MarketData\Deribit\research\cs04a_feature_store
.............                                                            [100%]
13 passed in 6.37s




## 1 — Input manifest and peer market preparation

BTC uses ETH as the separate futures peer; ETH/SOL/XRP use BTC. Same-time peer readings only. No forward filling. For production the parent directory contains existing local Binance 1-minute Parquet files. Synthetic data use a temporary directory and **cannot establish market evidence**.

In [2]:
ASSETS=('BTC',) if SYNTHETIC else tuple(cfg.assets)
loaded={};excluded={}
if SYNTHETIC:
    from tests.test_candle_streak_feature_matrix import simulated_inputs
    minute,spot,close,f=simulated_inputs(days=69,seed=2606)
    loaded['BTC']=(spot,close,f)
    PEERS={'BTC':f[['state_time','futures_r5','flow_5m','flow_valid']].copy()}
else:
    PEERS={}
    for peer in ('BTC','ETH'):
        path=BINANCE/f'{peer}USDT_um_1m.parquet'
        assert path.exists(), f'Missing peer futures file {path}'
        z=pd.read_parquet(path,columns=['open_time','open','high','low','close','quote_volume','taker_buy_quote_volume','trades'])
        z=z.loc[pd.to_datetime(z.open_time,utc=True)<pd.Timestamp(cfg.cutoff_utc)]
        f=fm.minute_to_five(z)
        PEERS[peer]=f[['state_time','futures_r5','flow_5m','flow_valid']].copy()
        print(peer,'peer bars:',len(f));del z,f;gc.collect()
print('Cross-asset peers prepared:',list(PEERS))

BTC peer bars: 213408
ETH peer bars: 213408
Cross-asset peers prepared: ['BTC', 'ETH']


## 2 — Iterate each asset and checkpoint the 175-family search

The full run can be computationally substantial. It writes a resumable *per-asset* checkpoint after each model fit. These checkpoint CSVs are aggregate model scores, never per-origin predictions. `CS04A_QUICK=1` runs the first 10 combinations only; use **without QUICK** for the registered complete study. The full run should be the only one cited as CS04-A results.

In [3]:
all_results=[];coverage=[];fold_metadata=[];quality=[]
for asset in ASSETS:
    stamp=time.perf_counter()
    if SYNTHETIC:
        sp,close,ff=loaded[asset]
    else:
        path=BINANCE/f'{asset}USDT_um_1m.parquet'
        try:
            if not path.exists():raise FileNotFoundError(path)
            raw=pd.read_parquet(path,columns=['open_time','open','high','low','close','quote_volume','taker_buy_quote_volume','trades'])
            raw=raw.loc[pd.to_datetime(raw.open_time,utc=True)<pd.Timestamp(cfg.cutoff_utc)]
            ff=fm.minute_to_five(raw);del raw;gc.collect()
            ret,missing=vd14.load_returns(SPOT,(asset,),end=pd.Timestamp(cfg.cutoff_utc),min_days=365)
            sp=ret.loc[ret.underlying.eq(asset),['state_time','return_5m']]
            close=pd.read_csv(SPOT/f'{asset}.csv.gz',usecols=['state_time','close'])
            close.state_time=pd.to_datetime(close.state_time,utc=True)
            close=close.loc[close.state_time<pd.Timestamp(cfg.cutoff_utc)]
        except Exception as exc:
            excluded[asset]=repr(exc);print(asset,'SKIPPED',exc);continue
    peer_name='ETH' if asset=='BTC' else 'BTC'
    peer=PEERS.get(peer_name,PEERS.get('BTC') if SYNTHETIC else None)
    try:
        panel=fm.attach_features(sp,ff,close,peer)
        panel=fm.attach_targets(panel)
        panel['asset']=asset
        panel['date']=panel.state_time.dt.floor('D')
        days=fm.eligible_complete_days(panel)
        quality.append({'asset':asset,'spot_rows':len(sp),'futures_buckets':len(ff),
                        'feature_ready':int(panel.feature_ready.sum()),
                        'complete_days':len(days),'first_day':str(days.min()),'last_day':str(days.max())})
        if len(days)<65:raise ValueError(f'Only {len(days)} eligible complete days')
        # This feature store stays LOCAL: never push individual labeled price records to GitHub.
        fields=list(dict.fromkeys(['asset','state_time','date','return_5m','spot_close','sign','length',
              'feature_ready','all_targets_ready']+list(fm.BASE)+
              [c for family in fm.FAMILY_NAMES for c in fm.FAMILY_COLUMNS[family]]+
              [f'{prefix}_{h}m' for h in fm.HORIZONS for prefix in ('ret','abs','sq','upside','downside','path_rv','path_abs_sum')]))
        if SYNTHETIC:
            panel.loc[:,fields].to_pickle(LOCAL/f'{asset}_features_and_forward_magnitude.pickle')
        else:
            panel.loc[:,fields].to_parquet(LOCAL/f'{asset}_features_and_forward_magnitude.parquet',index=False)
        checkpoint=OUT/f'04a_checkpoint_{asset}.csv'
        result,folds,full_days=fm.scan_asset(panel,cfg,max_combos=10 if QUICK else None,
                                              checkpoint=checkpoint)
        all_results.append(result)
        coverage.append({'asset':asset,'n_overlap_complete_days':len(full_days),
            'n_combinations_including_base':result.combination.nunique(),
            'n_fold_fits':len(result),'n_heldout_days':sum(len(f['validation_days']) for f in folds)})
        fold_metadata.append(fm.coverage_manifest(panel,full_days,folds,cfg))
        print(asset,'completed:',len(result),'fits,',len(full_days),'complete days,',
              'local targets saved, seconds:',round(time.perf_counter()-stamp,1))
    except Exception as exc:
        excluded[asset]=repr(exc);print(asset,'FAILED',type(exc).__name__,exc)
        if not SYNTHETIC:raise
    finally:
        del sp,close,ff
        if 'panel' in locals():del panel
        gc.collect()
assert all_results,'No eligible assets completed'
raw_scores=pd.concat(all_results,ignore_index=True)
assert not raw_scores[['asset','fold','combination']].duplicated().any()
print('Finished scoring fits:',len(raw_scores))
display(pd.DataFrame(quality));display(pd.DataFrame(coverage))

BTC completed: 528 fits, 736 complete days, local targets saved, seconds: 645.7
ETH completed: 528 fits, 736 complete days, local targets saved, seconds: 526.4
SOL completed: 528 fits, 736 complete days, local targets saved, seconds: 436.4
XRP completed: 528 fits, 736 complete days, local targets saved, seconds: 447.7
Finished scoring fits: 2112


,asset,spot_rows,futures_buckets,feature_ready,complete_days,first_day,last_day
0,BTC,221761,213408,213071,736,2024-10-02 00:00:00+00:00,2026-10-10 00:00:00+00:00
1,ETH,221761,213408,213071,736,2024-10-02 00:00:00+00:00,2026-10-10 00:00:00+00:00
2,SOL,221761,213408,213071,736,2024-10-02 00:00:00+00:00,2026-10-10 00:00:00+00:00
3,XRP,221761,213408,213071,736,2024-10-02 00:00:00+00:00,2026-10-10 00:00:00+00:00


,asset,n_overlap_complete_days,n_combinations_including_base,n_fold_fits,n_heldout_days
0,BTC,736,176,528,141
1,ETH,736,176,528,141
2,SOL,736,176,528,141
3,XRP,736,176,528,141


## 3 — Rank only incremental out-of-fold forecast losses

All candidate deltas reference the common model with **exactly the same asset, fold and evaluation dates**. Negative deltas improve probabilistic direction or mean-absolute-net-return MSE. Ranking is exploratory and unadjusted for selection bias. Cross-asset averages are diagnostic and may conceal asset-specific differences.

In [4]:
ranked=fm.rank_combinations(raw_scores)
print('Best three-class feature combinations (discovery only)')
display(ranked.head(25)[['combination','family_count','n_asset_folds','delta_threeclass_logloss',
        'delta_binary_logloss','delta_abs_5m_mse_bps2','delta_abs_15m_mse_bps2',
        'delta_abs_30m_mse_bps2','direction_better_folds','magnitude_5m_better_folds']])
print('Best next-5m absolute magnitude combinations (discovery only)')
display(ranked.sort_values('delta_abs_5m_mse_bps2').head(20)[['combination','delta_threeclass_logloss',
         'delta_abs_5m_mse_bps2','delta_abs_15m_mse_bps2','delta_abs_30m_mse_bps2']])

Best three-class feature combinations (discovery only)


,combination,family_count,n_asset_folds,delta_threeclass_logloss,delta_binary_logloss,delta_abs_5m_mse_bps2,delta_abs_15m_mse_bps2,delta_abs_30m_mse_bps2,direction_better_folds,magnitude_5m_better_folds
0,momentum+trade_activity+spot_futures_basis,3,12,-0.001196,-0.000522,-3.891202,-10.137946,-15.433833,10,11
1,momentum+candle_geometry+trade_activity,3,12,-0.001147,-0.000490,-10.952239,-27.714987,-44.850562,11,12
2,momentum+minute_path+trade_activity,3,12,-0.001107,-0.000487,-6.092584,-14.682268,-26.827463,10,11
3,minute_path+trade_activity+spot_futures_basis,3,12,-0.001094,-0.000430,-6.543352,-14.835486,-28.410479,11,12
4,trade_activity+flow_price_impact+spot_futures_...,3,12,-0.001061,-0.000348,-2.728622,-6.292702,-10.124256,9,11
5,candle_geometry+trade_activity+spot_futures_basis,3,12,-0.001057,-0.000360,-10.892586,-26.904606,-43.856443,11,12
6,signed_flow+trade_activity+spot_futures_basis,3,12,-0.001018,-0.000331,-2.810502,-6.404690,-10.054967,11,11
7,momentum+signed_flow+trade_activity,3,12,-0.000962,-0.000311,-3.475905,-9.227399,-16.613226,10,10
8,trade_activity+spot_futures_basis+peer_market,3,12,-0.000956,-0.000279,-2.752321,-6.728854,-11.200744,9,10
9,momentum+trade_activity+flow_price_impact,3,12,-0.000928,-0.000254,-3.521007,-9.321478,-17.076060,10,10


Best next-5m absolute magnitude combinations (discovery only)


,combination,delta_threeclass_logloss,delta_abs_5m_mse_bps2,delta_abs_15m_mse_bps2,delta_abs_30m_mse_bps2
68,momentum+candle_geometry+utc_calendar,-0.000500,-11.256817,-28.529924,-46.507294
55,momentum+candle_geometry+flow_price_impact,-0.000561,-11.153541,-27.772722,-44.286890
49,momentum+candle_geometry+signed_flow,-0.000589,-11.146826,-27.731429,-44.200898
117,candle_geometry+signed_flow+utc_calendar,-0.000211,-11.099444,-27.392470,-45.309206
135,candle_geometry+flow_price_impact+utc_calendar,-0.000130,-11.081470,-27.267176,-45.099847
43,momentum+candle_geometry+spot_futures_basis,-0.000621,-11.080138,-27.584044,-44.111844
57,momentum+candle_geometry+flow_timing,-0.000557,-11.042147,-27.538324,-43.642946
58,momentum+candle_geometry,-0.000555,-11.041132,-27.549535,-43.647132
62,momentum+candle_geometry+peer_market,-0.000521,-11.026169,-27.720755,-44.304890
144,candle_geometry+flow_timing+utc_calendar,-0.000119,-10.972365,-27.075612,-44.631450


## 4 — Save **aggregate** rankings, folds and magnitude diagnostics

Only aggregate scores enter GitHub. The local compressed labeled feature panel contains return magnitudes at all three horizons but **must not be committed**. Do not use the previously read CS03/CS03b TEST period to choose a candidate.

In [5]:
raw_scores.to_csv(OUT/'04a_fold_scores.csv',index=False)
ranked.to_csv(OUT/'04a_ranked_combinations.csv',index=False)
pd.DataFrame(coverage).to_csv(OUT/'04a_coverage.csv',index=False)
pd.DataFrame(quality).to_csv(OUT/'04a_quality.csv',index=False)
pd.DataFrame([{'family':f,'n_features':len(cols),'features':' | '.join(cols)}
             for f,cols in fm.FAMILY_COLUMNS.items()]).to_csv(OUT/'04a_feature_catalog.csv',index=False)
manifest={'module':'CS04-A','registration':fm.registration(),
         'synthetic':SYNTHETIC,'quick':QUICK,'assets_evaluated':[x['asset'] for x in quality],
         'excluded':excluded,'folds':fold_metadata,'model_count':int(len(raw_scores)),
         'feature_combination_count':int(ranked.combination.nunique()),
         'target_labels_locally_saved':str(LOCAL),
         'local_target_columns':[f'{prefix}_{h}m' for h in fm.HORIZONS for prefix in ('ret','abs','sq','upside','downside','path_rv','path_abs_sum')],
         'previously_inspected_legacy_test_used_for_selection':False,
         'AM_T_HAR_Blend_modified':False,
         'disclaimer':'Exploratory only; do not report rank #1 as independent confirmation.'}
(OUT/'04a_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved six aggregate outputs + checkpoints:',OUT)
print('Source of reusable magnitude labels (LOCAL ONLY):',LOCAL)

Saved six aggregate outputs + checkpoints: D:\Projects\deribit_market_data\notebooks\Candle_Streaks\04a
Source of reusable magnitude labels (LOCAL ONLY): D:\MarketData\Deribit\research\cs04a_feature_store


## Interpretation and next stage

- The 175 additive subset candidates are a **discovery scan**, not 175 independent confirmations. Future model selection must account for correlated alternatives and multiple comparisons, with freshly collected evidence.
- `binary_logloss` conditions on next nonzero 5-minute candle and is **not** a deployable forecast by itself. Primary probabilistic output includes DOWN/ZERO/UP.
- The magnitude targets are **signed log net returns**, their absolute values, squares and positive/negative magnitudes over (t,t+h] for h=5,15,30 minutes; the model currently forecasts only the **conditional mean absolute net move**, not tail quantiles or a calibrated density. No Student-t parameters are changed.
- **No current Deribit OBI** is incorporated; only local historical Binance spot prices and USD-M one-minute executions/price bars are used.
- `04a_ranked_combinations.csv` and `04a_fold_scores.csv` are GitHub-shareable aggregate results. `DERIBIT_DATA_ROOT/research/cs04a_feature_store/*.parquet` is private local cache with per-origin futures labels and should **not** be pushed.
- Next stage should freeze shortlisted candidates and evaluate on *newly observed data*, compare against sign-only, CS03b, and AM-T-HAR-Blend at exactly matching horizons, then assess proper density NLL, magnitude calibration, tails, and execution costs.